<a href="https://colab.research.google.com/github/pierrot73/GenAIBootCamp/blob/Bootcamp/Week7_Day1_DC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
# Installe les bibliothèques nécessaires

!pip install -q transformers datasets evaluate scikit-learn accelerate tensorflow
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [35]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [36]:
# loading...
TRAIN_CSV = f"/content/drive/MyDrive/Colab Notebooks/train.csv"
TEST_CSV  = f"/content/drive/MyDrive/Colab Notebooks/test.csv"

train_df = pd.read_csv(TRAIN_CSV)
print("\nTrain preview:\n", train_df.head())
print("\nLabel distribution:\n", train_df['label'].value_counts())
test_df = pd.read_csv(TEST_CSV) if os.path.exists(TEST_CSV) else None


Train preview:
            id                                            premise  \
0  5130fd2cb5  and these comments were considered in formulat...   
1  5b72532a0b  These are issues that we wrestle with in pract...   
2  3931fbe82a  Des petites choses comme celles-là font une di...   
3  5622f0c60b  you know they can't really defend themselves l...   
4  86aaa48b45  ในการเล่นบทบาทสมมุติก็เช่นกัน โอกาสที่จะได้แสด...   

                                          hypothesis lang_abv language  label  
0  The rules developed in the interim were put to...       en  English      0  
1  Practice groups are not permitted to work on t...       en  English      2  
2              J'essayais d'accomplir quelque chose.       fr   French      0  
3  They can't defend themselves because of their ...       en  English      0  
4    เด็กสามารถเห็นได้ว่าชาติพันธุ์แตกต่างกันอย่างไร       th     Thai      1  

Label distribution:
 label
0    4176
2    4064
1    3880
Name: count, dtype: int64


In [37]:
# Distribution des labels
print(train_df['label'].value_counts())
import random

idx = random.randint(0, len(train_df) - 1)
print("Prémisse :", train_df.loc[idx, 'premise'])
print("Hypothèse :", train_df.loc[idx, 'hypothesis'])
print("Label :", train_df.loc[idx, 'label'])


label
0    4176
2    4064
1    3880
Name: count, dtype: int64
Prémisse : Một người khởi xướng trứ danh về văn hóa dân gian Mexico Tây Ban Nha ở Miền Tây Nam và California chính là Charles F. Lummis (1859-1928), một nhiếp ảnh gia tự học, nhà dân tộc học, nhà âm nhạc, nhà báo và là người sáng lập Bảo tàng Tây Nam ở Los Angeles.
Hypothèse : Charles F. Lewis không biết đọc.
Label : 2


In [38]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')

def bert_encode(premises, hypotheses, tokenizer, max_length=128):
    return tokenizer(
        premises.tolist(),
        hypotheses.tolist(),
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors='pt'
    )

train_encodings = bert_encode(train_df['premise'], train_df['hypothesis'], tokenizer)
test_encodings = bert_encode(test_df['premise'], test_df['hypothesis'], tokenizer)

# Convertir les labels
import torch
train_labels = torch.tensor(train_df['label'].values)


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

In [39]:
import torch.nn as nn
from transformers import BertModel

class BERTClassifier(nn.Module):
    def __init__(self):
        super(BERTClassifier, self).__init__()
        self.bert = BertModel.from_pretrained('bert-base-multilingual-cased')
        self.dropout = nn.Dropout(p=0.3)
        self.classifier = nn.Linear(self.bert.config.hidden_size, 3)

    def forward(self, input_ids, attention_mask, token_type_ids):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        pooled_output = outputs.pooler_output
        pooled_output = self.dropout(pooled_output)
        return self.classifier(pooled_output)


In [40]:
from torch.utils.data import TensorDataset, DataLoader

# Dataset
train_dataset = TensorDataset(
    train_encodings['input_ids'],
    train_encodings['attention_mask'],
    train_encodings['token_type_ids'],
    train_labels
)

# DataLoader
train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=16)

# Modèle
model = BERTClassifier()

# Optimiseur et perte
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

# GPU si dispo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)


model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

BERTClassifier(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwi

In [4]:
epochs = 3

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch in train_dataloader:
        input_ids, attention_mask, token_type_ids, labels = [b.to(device) for b in batch]

        outputs = model(input_ids, attention_mask, token_type_ids)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch + 1} | Loss: {total_loss / len(train_dataloader):.4f}")

NameError: name 'model' is not defined